In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv

load_dotenv()


True

In [6]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
#  You are importing two tools from LangChain.

#  HuggingFaceEndpoint
#  This is used to connect LangChain to a Hugging Face model/API
#  ChatHuggingFace
# This makes the Hugging Face model behave like a chat model that LangChain can work with.

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-32B",
    temperature=0.7,
    max_new_tokens=1000
)

model = ChatHuggingFace(llm=llm)


In [7]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str


In [ ]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [9]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [10]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)